In [ ]:
import csv
import random
from pathlib import Path

import torch
from torch.utils.data import Dataset
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
import json
import shutil
from datetime import datetime

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

In [ ]:
RESUME_CHECKPOINT_PATH = Path("/kaggle/input/models/danielebracoloni/secunda-model-26checkpoint/pytorch/default/1/secunda_slakh_8cb_448d_14l/secunda_slakh_emergency_last.pt")

In [ ]:
class SecundaAudioDataset(Dataset):
    def __init__(
        self,
        manifest_path,
        source_roots,
        split="train",
        stage=None,
        segment_seconds=5.0,
        frame_rate=75,
        fixed_start=False,
        seed=42,
        samples_per_track=1,
    ):
        self.manifest_path = Path(manifest_path)

        self.source_roots = {
            key: Path(value)
            for key, value in source_roots.items()
        }

        self.segment_frames = int(
            round(segment_seconds * frame_rate)
        )

        if self.segment_frames < 2:
            raise ValueError(
                "segment_seconds must produce at least 2 frames."
            )

        if samples_per_track < 1:
            raise ValueError(
                "samples_per_track must be at least 1."
            )

        self.fixed_start = fixed_start
        self.seed = seed
        self.epoch = 0
        self.samples_per_track = samples_per_track

        with open(
            self.manifest_path,
            newline="",
            encoding="utf-8",
        ) as file:
            all_rows = list(csv.DictReader(file))

        self.rows = [
            row
            for row in all_rows
            if row["split"] == split
            and row["split"] != "excluded"
            and (
                stage is None
                or row["stage"] == stage
            )
        ]

        if not self.rows:
            raise ValueError(
                f"No rows found for split={split}, stage={stage}"
            )

        self._validate_rows()

    def _validate_rows(self):
        for row in self.rows:
            source = row["source"]

            if source not in self.source_roots:
                raise KeyError(
                    f"No root configured for source={source}"
                )

            filename = Path(row["path"]).name
            tensor_path = self.source_roots[source] / filename

            if not tensor_path.exists():
                raise FileNotFoundError(tensor_path)

            if int(row["num_codebooks"]) != 32:
                raise ValueError(
                    f"{tensor_path}: expected 32 codebooks"
                )

            if int(row["frame_rate"]) != 75:
                raise ValueError(
                    f"{tensor_path}: expected 75 fps"
                )

            if float(row["bandwidth_kbps"]) != 24.0:
                raise ValueError(
                    f"{tensor_path}: expected 24 kbps"
                )

            if int(row["num_frames"]) < self.segment_frames:
                raise ValueError(
                    f"{tensor_path}: has only "
                    f"{row['num_frames']} frames, but needs "
                    f"{self.segment_frames}"
                )

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.rows) * self.samples_per_track

    def _path_for_row(self, row):
        source = row["source"]
        filename = Path(row["path"]).name
        return self.source_roots[source] / filename

    def _choose_start(self,row_index,crop_index,total_frames,):
        max_start = total_frames - self.segment_frames
    
        if max_start <= 0:
            return 0
    
        if self.fixed_start:
            # Validation:
            generator = random.Random(
                self.seed
                + 99_999_937
                + row_index * 10_000
                + crop_index
            )
        else:
            # Training:
            # deterministic but changes with epoch.
            generator = random.Random(
                self.seed
                + self.epoch * 1_000_000
                + row_index * 10_000
                + crop_index
            )
    
        return generator.randint(0, max_start)



    def __getitem__(self, index):
        row_index = index // self.samples_per_track
        crop_index = index % self.samples_per_track

        row = self.rows[row_index]
        tensor_path = self._path_for_row(row)

        data = torch.load(
            tensor_path,
            map_location="cpu",
        )

        tokens = data["tokens"].long()
        tension = data["tension"].float()
        combat_score = data["combat_score"].float()

        if tokens.ndim != 2:
            raise ValueError(
                f"{tensor_path}: tokens must be [n_q, T]"
            )

        total_frames = tokens.shape[-1]

        if total_frames != len(tension):
            raise ValueError(
                f"{tensor_path}: tokens/tension mismatch"
            )

        if total_frames != len(combat_score):
            raise ValueError(
                f"{tensor_path}: tokens/combat mismatch"
            )

        start = self._choose_start(
            row_index=row_index,
            crop_index=crop_index,
            total_frames=total_frames,
        )

        end = start + self.segment_frames

        return {
            "tokens": tokens[:NUM_CODEBOOKS, start:end],
            "tension": tension[start:end],
            "combat_score": combat_score[start:end],
            "source": row["source"],
            "track_id": row["track_id"],
            "path": row["path"],
            "start_frame": start,
            "crop_index": crop_index,
        }

In [ ]:
# frame codec 
NUM_CODEBOOKS = 8
CODEBOOK_SIZE = 1024
PAD_TOKEN_ID = CODEBOOK_SIZE
MODEL_VOCAB_SIZE = CODEBOOK_SIZE + 1


def validate_code_tensor(
    codes,
    num_codebooks=NUM_CODEBOOKS,
    codebook_size=CODEBOOK_SIZE,
):
    if codes.ndim != 3:
        raise ValueError(
            "Expected codes shaped [batch, num_codebooks, time], "
            f"got {tuple(codes.shape)}"
        )

    if codes.shape[1] != num_codebooks:
        raise ValueError(
            f"Expected {num_codebooks} codebooks, "
            f"got {codes.shape[1]}"
        )

    if codes.numel() == 0:
        raise ValueError("Code tensor is empty.")

    minimum = int(codes.min())
    maximum = int(codes.max())

    if minimum < 0 or maximum >= codebook_size:
        raise ValueError(
            f"Code IDs must lie in [0, {codebook_size - 1}], "
            f"but found min={minimum}, max={maximum}."
        )


def validate_condition_tensor(condition, name, batch_size, time_steps):
    if condition.ndim != 2:
        raise ValueError(
            f"{name} must be shaped [batch, time], "
            f"got {tuple(condition.shape)}"
        )

    expected_shape = (batch_size, time_steps)

    if tuple(condition.shape) != expected_shape:
        raise ValueError(
            f"{name} must have shape {expected_shape}, "
            f"got {tuple(condition.shape)}"
        )


class CodebookSerializer:
    """
    Implements the diagonal delayed-codebook pattern.

    Input:
        codes: [B, Q, T]

    Output:
        delayed_codes: [B, Q, T + Q - 1]

    Codebook q is delayed by q positions:
        delayed_codes[:, q, q:q+T] = codes[:, q, :]

    Empty positions receive PAD_TOKEN_ID = 1024.
    """

    def __init__(
        self,
        num_codebooks=NUM_CODEBOOKS,
        codebook_size=CODEBOOK_SIZE,
        pad_token_id=PAD_TOKEN_ID,
    ):
        self.num_codebooks = num_codebooks
        self.codebook_size = codebook_size
        self.pad_token_id = pad_token_id

        if pad_token_id < codebook_size:
            raise ValueError(
                "PAD token must not overlap valid EnCodec IDs."
            )

    @property
    def model_vocab_size(self):
        return self.pad_token_id + 1

    def delay(self, codes):
        validate_code_tensor(
            codes,
            num_codebooks=self.num_codebooks,
            codebook_size=self.codebook_size,
        )

        batch_size, _, time_steps = codes.shape
        delayed_time = time_steps + self.num_codebooks - 1

        delayed_codes = torch.full(
            (
                batch_size,
                self.num_codebooks,
                delayed_time,
            ),
            fill_value=self.pad_token_id,
            dtype=codes.dtype,
            device=codes.device,
        )

        for codebook_index in range(self.num_codebooks):
            start = codebook_index
            end = start + time_steps

            delayed_codes[
                :,
                codebook_index,
                start:end,
            ] = codes[:, codebook_index, :]

        return delayed_codes

    def undelay(self, delayed_codes):
        if delayed_codes.ndim != 3:
            raise ValueError(
                "Expected delayed codes shaped [B, Q, S], "
                f"got {tuple(delayed_codes.shape)}"
            )

        batch_size, found_codebooks, delayed_time = delayed_codes.shape

        if found_codebooks != self.num_codebooks:
            raise ValueError(
                f"Expected {self.num_codebooks} codebooks, "
                f"got {found_codebooks}."
            )

        original_time = delayed_time - self.num_codebooks + 1

        if original_time <= 0:
            raise ValueError(
                "Delayed sequence is too short to recover codes."
            )

        recovered_codes = torch.empty(
            (
                batch_size,
                self.num_codebooks,
                original_time,
            ),
            dtype=delayed_codes.dtype,
            device=delayed_codes.device,
        )

        for codebook_index in range(self.num_codebooks):
            start = codebook_index
            end = start + original_time

            recovered_codes[
                :,
                codebook_index,
                :,
            ] = delayed_codes[
                :,
                codebook_index,
                start:end,
            ]

        validate_code_tensor(
            recovered_codes,
            num_codebooks=self.num_codebooks,
            codebook_size=self.codebook_size,
        )

        return recovered_codes


def delay_conditions_by_codebook(tension, combat_score, num_codebooks=NUM_CODEBOOKS, pad_value=0.0):
    """
    Aligns time-varying controls to the staggered codebook targets.
    
    Inputs:
        tension, combat_score: [B, T]
        
    Outputs:
        delayed_tension: [B, Q, T + Q - 1]
        delayed_combat:  [B, Q, T + Q - 1]
        valid_mask:      [B, Q, T + Q - 1] (True where real audio exists)
    """
    batch_size, original_time = tension.shape
    delayed_time = original_time + num_codebooks - 1

    # Initialize with a pad value (e.g., 0.0). These positions will be ignored in the loss.
    delayed_tension = torch.full(
        (batch_size, num_codebooks, delayed_time), 
        pad_value, dtype=tension.dtype, device=tension.device
    )
    delayed_combat = torch.full(
        (batch_size, num_codebooks, delayed_time), 
        pad_value, dtype=combat_score.dtype, device=combat_score.device
    )
    valid_mask = torch.zeros(
        (batch_size, num_codebooks, delayed_time), 
        dtype=torch.bool, device=tension.device
    )

    for q in range(num_codebooks):
        start = q
        end = q + original_time
        delayed_tension[:, q, start:end] = tension
        delayed_combat[:, q, start:end] = combat_score
        valid_mask[:, q, start:end] = True

    return delayed_tension, delayed_combat, valid_mask


def prepare_next_frame_batch(batch, serializer=None):
    """
    Updated to return codebook-aligned conditions.
    """
    if serializer is None:
        serializer = CodebookSerializer()

    tokens = batch["tokens"].long()
    tension = batch["tension"].float()
    combat_score = batch["combat_score"].float()

    delayed_codes = serializer.delay(tokens)
    delayed_tension, delayed_combat, valid_mask = delay_conditions_by_codebook(
        tension, combat_score, serializer.num_codebooks
    )

    # Autoregressive shift: inputs get 0 to end-1, targets get 1 to end
    input_codes = delayed_codes[:, :, :-1]
    target_codes = delayed_codes[:, :, 1:]
    
    # Conditions align with the TARGET frame, so we slice them 1 to end as well
    target_tension = delayed_tension[:, :, 1:]
    target_combat = delayed_combat[:, :, 1:]
    target_mask = valid_mask[:, :, 1:]

    return {
        "input_codes": input_codes,
        "target_codes": target_codes,
        "target_tension": target_tension,
        "target_combat": target_combat,
        "target_mask": target_mask
    }

class FrameCodebookEmbedding(nn.Module):
    """
    Converts delayed [B, Q, S] IDs into [B, S, D] frame representations.

    PAD_TOKEN_ID has a zero embedding and does not contribute to the
    summed frame representation.
    """

    def __init__(
        self,
        embedding_dim,
        num_codebooks=NUM_CODEBOOKS,
        codebook_size=CODEBOOK_SIZE,
        pad_token_id=PAD_TOKEN_ID,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.num_codebooks = num_codebooks
        self.codebook_size = codebook_size
        self.pad_token_id = pad_token_id
        self.model_vocab_size = pad_token_id + 1

        self.embeddings = nn.ModuleList(
            [
                nn.Embedding(
                    num_embeddings=self.model_vocab_size,
                    embedding_dim=embedding_dim,
                    padding_idx=pad_token_id,
                )
                for _ in range(num_codebooks)
            ]
        )

    def forward(self, delayed_codes):
        if delayed_codes.ndim != 3:
            raise ValueError(
                "Expected delayed codes shaped [B, Q, S], "
                f"got {tuple(delayed_codes.shape)}"
            )

        if delayed_codes.shape[1] != self.num_codebooks:
            raise ValueError(
                f"Expected {self.num_codebooks} codebooks, "
                f"got {delayed_codes.shape[1]}."
            )

        minimum = int(delayed_codes.min())
        maximum = int(delayed_codes.max())

        if minimum < 0 or maximum > self.pad_token_id:
            raise ValueError(
                f"Delayed IDs must lie in [0, {self.pad_token_id}], "
                f"but found min={minimum}, max={maximum}."
            )

        code_embeddings = []

        for codebook_index, embedding in enumerate(self.embeddings):
            code_ids = delayed_codes[:, codebook_index, :]
            code_embeddings.append(embedding(code_ids))

        stacked = torch.stack(
            code_embeddings,
            dim=2,
        )

        frame_embeddings = stacked.sum(dim=2)
        frame_embeddings = frame_embeddings / (
            self.num_codebooks ** 0.5
        )

        return frame_embeddings


def multicodebook_cross_entropy_weighted(
    logits: torch.Tensor,
    target_codes: torch.Tensor,
    pad_token_id: int,
    loss_weights: torch.Tensor,
) -> torch.Tensor:
    """
    Weighted multi-codebook cross-entropy.

    logits: B, Q, S, V (V = CODEBOOK_SIZE+1)
    target_codes: B, Q, S, with EnCodec IDs [0..CODEBOOK_SIZE-1] or PAD_TOKEN_ID.
    pad_token_id: ID to ignore in loss.
    loss_weights: Q, weights per codebook.
    """
    if logits.ndim != 4:
        raise ValueError(f"Expected logits shaped (B,Q,S,V), got {logits.shape}")
    if target_codes.ndim != 3:
        raise ValueError(f"Expected targets shaped (B,Q,S), got {target_codes.shape}")
    if logits.shape[0] != target_codes.shape[0] or logits.shape[1] != target_codes.shape[1] or logits.shape[2] != target_codes.shape[2]:
        raise ValueError(f"logits and targets must agree on B,Q,S: {logits.shape} vs {target_codes.shape}")
    if logits.shape[1] != loss_weights.shape[0]:
        raise ValueError(f"loss_weights must have length Q={logits.shape[1]}, got {loss_weights.shape[0]}")

    B, Q, S, V = logits.shape

    # Flatten over time per codebook: treat each (q, t) as a token position.
    logits_flat = logits.reshape(B * Q * S, V)
    targets_flat = target_codes.reshape(B * Q * S)

    # Mask out PAD positions
    non_pad_mask = (targets_flat != pad_token_id)
    if non_pad_mask.sum() == 0:
        # No valid positions anywhere -> zero loss
        return logits.new_zeros(())

    logits_valid = logits_flat[non_pad_mask]
    targets_valid = targets_flat[non_pad_mask]

    # Cross-entropy over valid positions
    ce_per_pos = F.cross_entropy(
        logits_valid,
        targets_valid,
        reduction="none",
    )

    # Map positions back to (q, t) to apply per-codebook weights
    pos_idx = torch.nonzero(non_pad_mask, as_tuple=False).squeeze(1)
    q_idx = (pos_idx // S) % Q  # since we flattened as B,Q,S

    # Apply weights: each position gets the weight of its codebook q
    weights_per_pos = loss_weights[q_idx]

    weighted_ce = ce_per_pos * weights_per_pos
    # Normalize by sum of weights to keep the scale reasonable
    loss = weighted_ce.sum() / weights_per_pos.sum()
    return loss

In [ ]:
class ConditionMLP(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, tension, combat_score):
        if tension.shape != combat_score.shape:
            raise ValueError(
                "tension and combat_score must have identical shapes."
            )

        conditions = torch.stack(
            [tension, combat_score],
            dim=-1,
        )

        return self.network(conditions)


class SecundaTransformer(nn.Module):
    def __init__(
        self,
        embedding_dim=128,
        num_layers=2,
        num_heads=4,
        feedforward_dim=512,
        dropout=0.0,
        max_sequence_length=256,
        num_codebooks=NUM_CODEBOOKS,
        codebook_size=CODEBOOK_SIZE,
        pad_token_id=PAD_TOKEN_ID,
    ):
        super().__init__()

        if embedding_dim % num_heads != 0:
            raise ValueError(
                "embedding_dim must be divisible by num_heads."
            )

        self.embedding_dim = embedding_dim
        self.num_codebooks = num_codebooks
        self.codebook_size = codebook_size
        self.pad_token_id = pad_token_id
        self.model_vocab_size = MODEL_VOCAB_SIZE
        self.max_sequence_length = max_sequence_length

        self.codebook_embedding = FrameCodebookEmbedding(
            embedding_dim=embedding_dim,
            num_codebooks=num_codebooks,
            codebook_size=codebook_size,
            pad_token_id=pad_token_id,
        )

        self.position_embedding = nn.Embedding(
            max_sequence_length,
            embedding_dim,
        )

        self.input_condition_mlp = ConditionMLP(
            input_dim=2,
            hidden_dim=embedding_dim,
            output_dim=embedding_dim,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

        self.final_norm = nn.LayerNorm(embedding_dim)

        self.output_condition_mlp = ConditionMLP(
            input_dim=2,
            hidden_dim=embedding_dim,
            output_dim=2 * embedding_dim,
        )

        self.output_heads = nn.ModuleList(
            [
                nn.Linear(
                    embedding_dim,
                    self.model_vocab_size,
                )
                for _ in range(num_codebooks)
            ]
        )

    def _causal_mask(self, sequence_length, device):
        mask = torch.full(
            (sequence_length, sequence_length),
            float("-inf"),
            device=device,
        )

        return torch.triu(
            mask,
            diagonal=1,
        )

    def forward(
        self,
        input_codes,
        input_tension,
        input_combat,
        target_tension,
        target_combat,
    ):
        if input_codes.ndim != 3:
            raise ValueError(
                "input_codes must have shape [B, Q, S]."
            )

        batch_size, found_codebooks, sequence_length = input_codes.shape

        if found_codebooks != self.num_codebooks:
            raise ValueError(
                f"Expected {self.num_codebooks} codebooks, "
                f"got {found_codebooks}."
            )

        if sequence_length > self.max_sequence_length:
            raise ValueError(
                f"Sequence length {sequence_length} exceeds "
                f"max_sequence_length={self.max_sequence_length}."
            )

        if input_tension.shape != (batch_size, sequence_length):
            raise ValueError(
                "input_tension has incorrect shape: "
                f"{tuple(input_tension.shape)}"
            )

        if input_combat.shape != (batch_size, sequence_length):
            raise ValueError(
                "input_combat has incorrect shape: "
                f"{tuple(input_combat.shape)}"
            )

        expected_target_shape = (
            batch_size,
            self.num_codebooks,
            sequence_length,
        )

        if target_tension.shape != expected_target_shape:
            raise ValueError(
                "target_tension has incorrect shape: "
                f"{tuple(target_tension.shape)}"
            )

        if target_combat.shape != expected_target_shape:
            raise ValueError(
                "target_combat has incorrect shape: "
                f"{tuple(target_combat.shape)}"
            )

        frame_embeddings = self.codebook_embedding(
            input_codes
        )

        positions = torch.arange(
            sequence_length,
            device=input_codes.device,
        )

        positional_embeddings = self.position_embedding(
            positions
        ).unsqueeze(0)

        input_condition_embeddings = self.input_condition_mlp(
            input_tension,
            input_combat,
        )

        hidden = (
            frame_embeddings
            + positional_embeddings
            + input_condition_embeddings
        )

        hidden = self.transformer(
            hidden,
            mask=self._causal_mask(
                sequence_length,
                input_codes.device,
            ),
        )

        hidden = self.final_norm(hidden)

        codebook_logits = []

        for codebook_index in range(self.num_codebooks):
            codebook_tension = target_tension[
                :,
                codebook_index,
                :,
            ]

            codebook_combat = target_combat[
                :,
                codebook_index,
                :,
            ]

            film_parameters = self.output_condition_mlp(
                codebook_tension,
                codebook_combat,
            )

            gamma, beta = film_parameters.chunk(
                2,
                dim=-1,
            )

            conditioned_hidden = (
                (1.0 + gamma) * hidden
                + beta
            )

            logits = self.output_heads[codebook_index](
                conditioned_hidden
            )

            codebook_logits.append(logits)

        return torch.stack(
            codebook_logits,
            dim=1,
        )

In [ ]:
import math

def lr_at_step(step: int) -> float:
    # Warmup
    if step < WARMUP_STEPS:
        return PEAK_LR * step / max(1, WARMUP_STEPS)

    # Cosine decay from PEAK_LR to MIN_LR over [WARMUP_STEPS..MAX_STEPS]
    progress = min(max(step - WARMUP_STEPS, 0), MAX_STEPS - WARMUP_STEPS)
    decay_ratio = progress / max(1, MAX_STEPS - WARMUP_STEPS)
    cosine = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LR + (PEAK_LR - MIN_LR) * cosine

In [ ]:
# ── Baseline training configuration ───────────────────────────────────

# ── 8-codebook heavy training configuration ──

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/danielebracoloni/"
    "slakh-and-custom-csvs/all_tracks.csv"
)

SOURCE_ROOTS = {
    "slakh": Path(
        "/kaggle/input/datasets/danielebracoloni/"
        "tesors-slakh-375-checkpoint/tensors_final/train"
    ),
}

OUTPUT_DIR = Path(
    "/kaggle/working/secunda_slakh_8cb_448d_14l"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESUME_CHECKPOINT_PATH = Path(
    "/kaggle/input/models/danielebracoloni/secunda-model-26checkpoint/pytorch/default/1/secunda_slakh_8cb_448d_14l/secunda_slakh_emergency_last.pt"
)

assert RESUME_CHECKPOINT_PATH.exists(), RESUME_CHECKPOINT_PATH

FRAME_RATE = 75

# Keep 10 seconds for this capacity experiment.
SEGMENT_SECONDS = 10.0
SEGMENT_FRAMES = int(SEGMENT_SECONDS * FRAME_RATE)

# Training data / loader config
BATCH_SIZE = 24          # keep; you know this fits on Kaggle
NUM_WORKERS = 2
SAMPLES_PER_TRACK = 20   # more crops per track (was 8)
SAVE_EVERY_BATCHES = 25
NUM_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 5

# Model size (slightly larger, but still Kaggle-friendly)
EMBEDDING_DIM = 448      # was 384
NUM_LAYERS = 14          # was 12
NUM_HEADS = 8
FEEDFORWARD_DIM = 2304           # was 2048
DROPOUT = 0.10

# Codec / sequence config (must match your tensors)
NUM_CODEBOOKS = 8
CODEBOOK_SIZE = 1024       # valid codes: 0..1023
PAD_TOKEN_ID = 1024        # EnCodec PAD, must match your tensors
FRAME_RATE = 75            # Hz
SEGMENT_SECONDS = 10.0
SEGMENT_FRAMES = int(round(SEGMENT_SECONDS * FRAME_RATE))

assert SEGMENT_FRAMES >= 2, "segment must produce at least 2 frames"

# LR schedule constants (steps are computed later, after the loaders exist)
PEAK_LR = 2.0e-4           # higher than 1.5e-4
MIN_LR = 3.0e-5            # same LR floor
WARMUP_STEPS = 4_000       # longer warmup

WEIGHT_DECAY = 0.01
GRAD_CLIP_NORM = 1.0
MAX_SEQUENCE_LENGTH = SEGMENT_FRAMES + NUM_CODEBOOKS - 1

# Codebook loss weights: push Q0-Q3 hard, but stop neglecting Q4-Q7
CODEBOOK_LOSS_WEIGHTS = torch.tensor(
    [4.0, 3.2, 2.7, 2.2, 2.0, 2.0, 2.0, 2.0],
    dtype=torch.float32,
    device=device,
)

assert CODEBOOK_LOSS_WEIGHTS.numel() == NUM_CODEBOOKS
assert MANIFEST_PATH.exists(), (
    f"Manifest not found: {MANIFEST_PATH}"
)

assert SOURCE_ROOTS["slakh"].exists(), (
    f"Slakh tensors not found: {SOURCE_ROOTS['slakh']}"
)

print("Manifest:", MANIFEST_PATH)
print("Slakh root:", SOURCE_ROOTS["slakh"])
print("Segment frames:", SEGMENT_FRAMES)

train_dataset = SecundaAudioDataset(
    manifest_path=MANIFEST_PATH,
    source_roots=SOURCE_ROOTS,
    split="train",
    stage="pretrain",
    segment_seconds=SEGMENT_SECONDS,
    frame_rate=FRAME_RATE,
    fixed_start=False,
    seed=RANDOM_SEED,
    samples_per_track=SAMPLES_PER_TRACK,
)

val_dataset = SecundaAudioDataset(
    manifest_path=MANIFEST_PATH,
    source_roots=SOURCE_ROOTS,
    split="val",
    stage="pretrain",
    segment_seconds=SEGMENT_SECONDS,
    frame_rate=FRAME_RATE,
    fixed_start=True,
    seed=RANDOM_SEED,
    samples_per_track=1,
)
val_dataset.set_epoch(0)

print("Validation crops:")

starts = []

for index in range(12):
    item = val_dataset[index]

    start_frame = int(item["start_frame"])
    start_seconds = start_frame / FRAME_RATE

    starts.append(start_frame)

    print(
        f"{index:02d} | "
        f"{item['track_id']} | "
        f"frame={start_frame} | "
        f"time={start_seconds:.2f}s"
    )

assert any(start > 0 for start in starts), (
    "Validation still uses only frame-zero crops."
)

print("PASS: validation uses deterministic nonzero interior crops.")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

# Training schedule: compute steps only AFTER the loaders exist,
# so STEPS_PER_EPOCH reflects the real number of batches per epoch.
STEPS_PER_EPOCH = len(train_loader)
TOTAL_TRAINING_STEPS = NUM_EPOCHS * STEPS_PER_EPOCH

# MAX_STEPS must reflect the real number of training steps,
# not an arbitrary fixed number.
MAX_STEPS = TOTAL_TRAINING_STEPS

print("\nDataset sizes:")
print("Slakh source training tracks:", len(train_dataset.rows))
print("Training clips per epoch:", len(train_dataset))
print("Slakh validation tracks:", len(val_dataset))
print("Train batches per epoch:", STEPS_PER_EPOCH)
print("Validation batches per epoch:", len(val_loader))
print("Maximum training steps:", TOTAL_TRAINING_STEPS)
print("Warmup steps:", WARMUP_STEPS)

In [ ]:
import csv

train_tensor_count = len(
    list(SOURCE_ROOTS["slakh"].glob("train_*.pt"))
)

print("Physical Slakh train tensors:", train_tensor_count)

assert train_tensor_count == 800, (
    f"Expected 800 Slakh train tensors, found {train_tensor_count}. "
    "You may be mounted to the old/incomplete dataset version."
)

with open(MANIFEST_PATH, newline="", encoding="utf-8") as file:
    manifest_rows = list(csv.DictReader(file))

slakh_pretrain_rows = [
    row
    for row in manifest_rows
    if row["source"] == "slakh"
    and row["stage"] == "pretrain"
    and row["split"] in {"train", "val", "test"}
]

slakh_train_rows = [
    row for row in slakh_pretrain_rows
    if row["split"] == "train"
]

slakh_val_rows = [
    row for row in slakh_pretrain_rows
    if row["split"] == "val"
]

slakh_test_rows = [
    row for row in slakh_pretrain_rows
    if row["split"] == "test"
]

print("Manifest Slakh train rows:", len(slakh_train_rows))
print("Manifest Slakh val rows:", len(slakh_val_rows))
print("Manifest Slakh test rows:", len(slakh_test_rows))

assert len(slakh_train_rows) == 704
assert len(slakh_val_rows) == 96
assert len(slakh_test_rows) == 151

In [ ]:
import math

serializer = CodebookSerializer()

torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

model = SecundaTransformer(
    embedding_dim=EMBEDDING_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    feedforward_dim=FEEDFORWARD_DIM,
    dropout=DROPOUT,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    num_codebooks=NUM_CODEBOOKS,
    codebook_size=CODEBOOK_SIZE,
    pad_token_id=PAD_TOKEN_ID,
).to(device)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Model configuration:")
print("  codebooks:", NUM_CODEBOOKS)
print("  embedding dimension:", EMBEDDING_DIM)
print("  layers:", NUM_LAYERS)
print("  heads:", NUM_HEADS)
print("  feedforward dimension:", FEEDFORWARD_DIM)
print("  parameters:", f"{parameter_count:,}")
assert model.num_codebooks == NUM_CODEBOOKS
assert model.embedding_dim == EMBEDDING_DIM
assert model.max_sequence_length == MAX_SEQUENCE_LENGTH

decay_parameters = []
no_decay_parameters = []

for parameter_name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    is_bias = parameter_name.endswith("bias")
    is_norm = (
        "norm" in parameter_name.lower()
        or "layernorm" in parameter_name.lower()
    )

    if is_bias or is_norm:
        no_decay_parameters.append(parameter)
    else:
        decay_parameters.append(parameter)

optimizer = torch.optim.AdamW(
    [
        {
            "params": decay_parameters,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": no_decay_parameters,
            "weight_decay": 0.0,
        },
    ],
    lr=PEAK_LR,
)

# Carica il checkpoint di resume
resume_checkpoint = torch.load(
    RESUME_CHECKPOINT_PATH,
    map_location=device,
)

model.load_state_dict(
    resume_checkpoint["model_state_dict"],
    strict=True,
)

optimizer.load_state_dict(
    resume_checkpoint["optimizer_state_dict"]
)

# Stati di training da cui riprendere
start_epoch = int(resume_checkpoint["epoch"])
start_batch_index = int(resume_checkpoint["batch_index"])
global_step = int(resume_checkpoint["global_step"])

best_val_loss = float(
    resume_checkpoint.get("best_val_loss", float("inf"))
)

# Ripristina il learning rate coerente con global_step
current_lr = lr_at_step(global_step)
for param_group in optimizer.param_groups:
    param_group["lr"] = current_lr

print("Resuming training:")
print("  epoch:", start_epoch)
print("  batch index:", start_batch_index)
print("  global step:", global_step)
print("  best validation loss:", best_val_loss)
print("  optimizer LR:", optimizer.param_groups[0]["lr"])
print("Peak LR (PEAK_LR):", PEAK_LR)
print("Warmup steps:", WARMUP_STEPS)
print("Total training steps (MAX_STEPS):", MAX_STEPS)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(f"Model parameters: {parameter_count:,}")
print("Dropout:", DROPOUT)
print("Weight decay:", WEIGHT_DECAY)
print("Parameters with weight decay:", len(decay_parameters))
print("Parameters without weight decay:", len(no_decay_parameters))

In [ ]:
def prepare_batch_for_model(batch):
    prepared = prepare_next_frame_batch(
        batch=batch,
        serializer=serializer,
    )

    # Codebook 0 target at delayed step s maps to original frame s + 1.
    # This shared condition tells the Transformer the desired next
    # primary-codebook audio state.
    input_tension = prepared["target_tension"][
        :,
        0,
        :,
    ]

    input_combat = prepared["target_combat"][
        :,
        0,
        :,
    ]

    return {
        "input_codes": prepared["input_codes"].to(
            device,
            non_blocking=True,
        ),
        "target_codes": prepared["target_codes"].to(
            device,
            non_blocking=True,
        ),
        "input_tension": input_tension.to(
            device,
            non_blocking=True,
        ),
        "input_combat": input_combat.to(
            device,
            non_blocking=True,
        ),
        "target_tension": prepared["target_tension"].to(
            device,
            non_blocking=True,
        ),
        "target_combat": prepared["target_combat"].to(
            device,
            non_blocking=True,
        ),
    }


def multi_codebook_metrics(
    logits,
    target_codes,
):
    """
    Returns:
        total_loss: scalar tensor (weighted)
        per_codebook_loss: [NUM_CODEBOOKS] tensor (unweighted, for logging)
        per_codebook_accuracy: [NUM_CODEBOOKS] tensor

    PAD targets are excluded from every metric.
    """
    # Keep the original unweighted per-codebook losses for logging.
    per_codebook_loss = []
    per_codebook_accuracy = []

    codebook_losses = []  # will hold unweighted losses as tensors

    for codebook_index in range(NUM_CODEBOOKS):
        codebook_logits = logits[
            :,
            codebook_index,
            :,
            :,
        ]

        codebook_targets = target_codes[
            :,
            codebook_index,
            :,
        ]

        valid_mask = codebook_targets != PAD_TOKEN_ID

        if valid_mask.sum() == 0:
            nan_tensor = torch.tensor(
                float("nan"),
                device=logits.device,
            )
            per_codebook_loss.append(nan_tensor)
            per_codebook_accuracy.append(nan_tensor)
            codebook_losses.append(nan_tensor)
            continue

        codebook_loss = F.cross_entropy(
            codebook_logits[valid_mask],
            codebook_targets[valid_mask],
        )

        predictions = codebook_logits.argmax(dim=-1)

        codebook_accuracy = (
            predictions[valid_mask] ==
            codebook_targets[valid_mask]
        ).float().mean()

        per_codebook_loss.append(codebook_loss)
        per_codebook_accuracy.append(codebook_accuracy)
        codebook_losses.append(codebook_loss)

    # Stack unweighted losses for logging
    per_codebook_loss = torch.stack(per_codebook_loss)
    per_codebook_accuracy = torch.stack(per_codebook_accuracy)
    codebook_losses = torch.stack(codebook_losses)  # [Q]

    # Apply codebook weights to compute the optimized total_loss
    weights = CODEBOOK_LOSS_WEIGHTS[:NUM_CODEBOOKS].to(codebook_losses.device)

    # Ignore NaN codebooks (if any) in the weighted average
    valid_losses = ~torch.isnan(codebook_losses)
    if valid_losses.sum() == 0:
        total_loss = torch.tensor(
            float("nan"),
            device=logits.device,
        )
    else:
        w = weights[valid_losses]
        losses = codebook_losses[valid_losses]
        total_loss = (losses * w).sum() / w.sum()

    return total_loss, per_codebook_loss, per_codebook_accuracy

def compute_batch_metrics(batch):
    model_batch = prepare_batch_for_model(batch)

    logits = model(
        input_codes=model_batch["input_codes"],
        input_tension=model_batch["input_tension"],
        input_combat=model_batch["input_combat"],
        target_tension=model_batch["target_tension"],
        target_combat=model_batch["target_combat"],
    )

    return multi_codebook_metrics(
        logits=logits,
        target_codes=model_batch["target_codes"],
    )


def save_emergency_checkpoint(
    epoch,
    batch_index,
    global_step,
    running_loss,
):
    emergency_path = (
        OUTPUT_DIR /
        "secunda_slakh_emergency_last.pt"
    )

    torch.save(
        {
            "epoch": epoch,
            "batch_index": batch_index,
            "global_step": global_step,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "running_train_loss": running_loss,
            "config": {
                "frame_rate": FRAME_RATE,
                "segment_seconds": SEGMENT_SECONDS,
                "segment_frames": SEGMENT_FRAMES,
                "samples_per_track": SAMPLES_PER_TRACK,
                "batch_size": BATCH_SIZE,
                "embedding_dim": EMBEDDING_DIM,
                "num_layers": NUM_LAYERS,
                "num_heads": NUM_HEADS,
                "feedforward_dim": FEEDFORWARD_DIM,
                "dropout": DROPOUT,
                "learning_rate": PEAK_LR,
                "weight_decay": WEIGHT_DECAY,
                "random_seed": RANDOM_SEED,
                "training_stage": "slakh_pretrain_baseline",
            },
        },
        emergency_path,
    )

    return emergency_path


def train_one_epoch(epoch, global_step):
    model.train()

    train_dataset.set_epoch(epoch)

    total_loss = 0.0
    total_per_codebook_loss = torch.zeros(
        NUM_CODEBOOKS,
        device=device,
    )

    total_per_codebook_accuracy = torch.zeros(
        NUM_CODEBOOKS,
        device=device,
    )

    batch_count = 0

    for batch_index, batch in enumerate(train_loader, start=1):
        if epoch == start_epoch and batch_index <= start_batch_index:
            continue
        # Advance global step first
        global_step += 1

        # Update LR from our warmup + cosine schedule
        lr = lr_at_step(global_step)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        optimizer.zero_grad(set_to_none=True)

        (
            loss,
            per_codebook_loss,
            per_codebook_accuracy,
        ) = compute_batch_metrics(batch)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch}, "
                f"batch {batch_index}: {loss.detach().item()}"
            )

        loss.backward()

        gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_NORM,
        )

        optimizer.step()

        loss_value = loss.detach().item()

        total_loss += loss_value
        total_per_codebook_loss += (
            per_codebook_loss.detach()
        )

        total_per_codebook_accuracy += (
            per_codebook_accuracy.detach()
        )

        batch_count += 1

        running_loss = total_loss / batch_count
        current_lr = optimizer.param_groups[0]["lr"]

        if batch_index == 1 or batch_index % 25 == 0:
            print(
                f"Epoch {epoch:02d} | "
                f"batch {batch_index:03d}/{len(train_loader)} | "
                f"loss={loss_value:.4f} | "
                f"running={running_loss:.4f} | "
                f"lr={current_lr:.2e} | "
                f"grad={gradient_norm.detach().item():.4f}"
            )

        if batch_index % SAVE_EVERY_BATCHES == 0:
            emergency_path = save_emergency_checkpoint(
                epoch=epoch,
                batch_index=batch_index,
                global_step=global_step,
                running_loss=running_loss,
            )

            print(
                f"Emergency checkpoint saved: "
                f"{emergency_path.name}"
            )

    return {
        "loss": total_loss / batch_count,
        "per_codebook_loss": (
            total_per_codebook_loss /
            batch_count
        ).cpu().tolist(),
        "per_codebook_accuracy": (
            total_per_codebook_accuracy /
            batch_count
        ).cpu().tolist(),
        "global_step": global_step,
    }




@torch.no_grad()
def validate_one_epoch():
    model.eval()

    # Fixed deterministic validation crops across all epochs.
    val_dataset.set_epoch(0)

    total_loss = 0.0
    total_per_codebook_loss = torch.zeros(
        NUM_CODEBOOKS,
        device=device,
    )

    total_per_codebook_accuracy = torch.zeros(
        NUM_CODEBOOKS,
        device=device,
    )

    batch_count = 0

    for batch in val_loader:
        (
            loss,
            per_codebook_loss,
            per_codebook_accuracy,
        ) = compute_batch_metrics(batch)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite validation loss: "
                f"{loss.detach().item()}"
            )

        total_loss += loss.detach().item()

        total_per_codebook_loss += (
            per_codebook_loss.detach()
        )

        total_per_codebook_accuracy += (
            per_codebook_accuracy.detach()
        )

        batch_count += 1

    return {
        "loss": total_loss / batch_count,
        "per_codebook_loss": (
            total_per_codebook_loss /
            batch_count
        ).cpu().tolist(),
        "per_codebook_accuracy": (
            total_per_codebook_accuracy /
            batch_count
        ).cpu().tolist(),
    }


def save_checkpoint(
    checkpoint_path,
    epoch,
    global_step,
    train_metrics,
    val_metrics,
    best_val_loss,
    history,
):
    checkpoint = {
        "epoch": epoch,
        "global_step": global_step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "best_val_loss": best_val_loss,
        "history": history,
        "config": {
            "frame_rate": FRAME_RATE,
            "segment_seconds": SEGMENT_SECONDS,
            "segment_frames": SEGMENT_FRAMES,
            "samples_per_track": SAMPLES_PER_TRACK,
            "batch_size": BATCH_SIZE,
            "embedding_dim": EMBEDDING_DIM,
            "num_layers": NUM_LAYERS,
            "num_heads": NUM_HEADS,
            "feedforward_dim": FEEDFORWARD_DIM,
            "dropout": DROPOUT,
            "max_sequence_length": MAX_SEQUENCE_LENGTH,
            "learning_rate": PEAK_LR,
            "weight_decay": WEIGHT_DECAY,
            "grad_clip_norm": GRAD_CLIP_NORM,
            "warmup_steps": WARMUP_STEPS,
            "total_training_steps": MAX_STEPS,
            "random_seed": RANDOM_SEED,
            "training_stage": "slakh_pretrain_8cb_384d_12l",
            "num_codebooks": NUM_CODEBOOKS,
            "codebook_loss_weights": (
                CODEBOOK_LOSS_WEIGHTS.detach().cpu().tolist()
            ),
        },
    }

    torch.save(checkpoint, checkpoint_path)

In [ ]:
# ── Heavy-model smoke test ──

model.eval()

smoke_batch = next(iter(train_loader))
smoke_prepared = prepare_batch_for_model(smoke_batch)

print("Smoke-test input codes:",
      tuple(smoke_prepared["input_codes"].shape))
print("Smoke-test target codes:",
      tuple(smoke_prepared["target_codes"].shape))

with torch.no_grad():
    smoke_logits = model(
        input_codes=smoke_prepared["input_codes"],
        input_tension=smoke_prepared["input_tension"],
        input_combat=smoke_prepared["input_combat"],
        target_tension=smoke_prepared["target_tension"],
        target_combat=smoke_prepared["target_combat"],
    )

print("Smoke-test logits:", tuple(smoke_logits.shape))

assert smoke_logits.shape[0] == BATCH_SIZE
assert smoke_logits.shape[1] == NUM_CODEBOOKS
assert smoke_logits.shape[2] == (
    SEGMENT_FRAMES + NUM_CODEBOOKS - 2
)
assert smoke_logits.shape[3] == MODEL_VOCAB_SIZE

smoke_loss, smoke_q_loss, smoke_q_acc = (
    multi_codebook_metrics(
        logits=smoke_logits,
        target_codes=smoke_prepared["target_codes"],
    )
)

print("Smoke-test loss:", float(smoke_loss))
print("Smoke-test per-codebook losses:",
      smoke_q_loss.detach().cpu().tolist())

del smoke_logits, smoke_prepared, smoke_batch
torch.cuda.empty_cache()

print("Heavy-model smoke test: PASS")

In [ ]:
print("CHECKPOINT TEST")
print("checkpoint epoch:", resume_checkpoint["epoch"])
print("checkpoint batch index:", resume_checkpoint["batch_index"])
print("checkpoint global step:", resume_checkpoint["global_step"])
print("start epoch:", start_epoch)
print("start batch index:", start_batch_index)
print("global step:", global_step)
print("optimizer LR:", optimizer.param_groups[0]["lr"])
# Questa riga va tolta perché scheduler non esiste più:
# print("scheduler last epoch:", scheduler.last_epoch)
print("expected LR at next step:", lr_at_step(global_step + 1))

In [ ]:
history = {
    "epoch": [],
    "global_step": [],
    "learning_rate": [],
    "train_loss": [],
    "val_loss": [],
    "train_per_codebook_loss": [],
    "val_per_codebook_loss": [],
    "train_per_codebook_accuracy": [],
    "val_per_codebook_accuracy": [],
}

bestvalloss = float(
    resume_checkpoint.get("bestvalloss", float("inf"))
)

epochswithoutimprovement = 0
globalstep = int(resume_checkpoint["global_step"])

best_checkpoint_path = (
    OUTPUT_DIR /
    "secunda_slakh8cb_best_final.pt"
)

last_checkpoint_path = (
    OUTPUT_DIR /
    "secunda_slakh8cb_last_final.pt"
)

history_path = (
    OUTPUT_DIR /
    "secunda_slakh8cb_training__final_history.json"
)

print("Starting Slakh baseline training.")
print("Output directory:", OUTPUT_DIR)
print("Maximum epochs:", NUM_EPOCHS)
print("Maximum steps:", TOTAL_TRAINING_STEPS)

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    print(f"\n{'=' * 80}")
    print(f"Epoch {epoch}/{NUM_EPOCHS}")
    print(f"{'=' * 80}")

    train_metrics = train_one_epoch(
        epoch=epoch,
        global_step=global_step,
    )

    global_step = train_metrics["global_step"]

    val_metrics = validate_one_epoch()

    train_loss = train_metrics["loss"]
    val_loss = val_metrics["loss"]

    current_lr = optimizer.param_groups[0]["lr"]

    history["epoch"].append(epoch)
    history["global_step"].append(global_step)
    history["learning_rate"].append(current_lr)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    history["train_per_codebook_loss"].append(
        train_metrics["per_codebook_loss"]
    )

    history["val_per_codebook_loss"].append(
        val_metrics["per_codebook_loss"]
    )

    history["train_per_codebook_accuracy"].append(
        train_metrics["per_codebook_accuracy"]
    )

    history["val_per_codebook_accuracy"].append(
        val_metrics["per_codebook_accuracy"]
    )

    improved = val_loss < best_val_loss

    print(
        f"\nEpoch {epoch:02d} summary | "
        f"step={global_step} | "
        f"train={train_loss:.4f} | "
        f"val={val_loss:.4f} | "
        f"best_val={min(best_val_loss, val_loss):.4f} | "
        f"lr={current_lr:.2e}"
    )

# Log a few representative codebooks: first, middle, last
    q0 = 0
    q_mid = NUM_CODEBOOKS // 2
    q_last = NUM_CODEBOOKS - 1
    
    print(
        "Validation codebook losses | "
        f"Q0={val_metrics['per_codebook_loss'][q0]:.4f} | "
        f"Q{q_mid}={val_metrics['per_codebook_loss'][q_mid]:.4f} | "
        f"Q{q_last}={val_metrics['per_codebook_loss'][q_last]:.4f}"
    )
    
    print(
        "Validation codebook accuracies | "
        f"Q0={val_metrics['per_codebook_accuracy'][q0]:.4f} | "
        f"Q{q_mid}={val_metrics['per_codebook_accuracy'][q_mid]:.4f} | "
        f"Q{q_last}={val_metrics['per_codebook_accuracy'][q_last]:.4f}"
    )
    print("Validation per-codebook metrics:")
    for q in range(NUM_CODEBOOKS):
        print(
            f"  Q{q}: "
            f"loss={val_metrics['per_codebook_loss'][q]:.4f}, "
            f"accuracy={val_metrics['per_codebook_accuracy'][q]:.4f}"
        )
    acc = torch.tensor(
    val_metrics["per_codebook_accuracy"],
    dtype=torch.float32,
    )
    
    print("Mean Q0-Q1 accuracy:", acc[:2].mean().item())
    print("Mean Q2-Q7 accuracy:", acc[2:].mean().item())
    
    if improved:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        save_checkpoint(
            checkpoint_path=best_checkpoint_path,
            epoch=epoch,
            global_step=global_step,
            train_metrics=train_metrics,
            val_metrics=val_metrics,
            best_val_loss=best_val_loss,
            history=history,
        )

        print(
            "Validation improved. "
            f"Saved best checkpoint: {best_checkpoint_path}"
        )

    else:
        epochs_without_improvement += 1

        print(
            "Validation did not improve. "
            f"Patience: {epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    save_checkpoint(
        checkpoint_path=last_checkpoint_path,
        epoch=epoch,
        global_step=global_step,
        train_metrics=train_metrics,
        val_metrics=val_metrics,
        best_val_loss=best_val_loss,
        history=history,
    )

    with open(history_path, "w") as file:
        json.dump(history, file, indent=2)

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            "\nEarly stopping: validation loss has not improved "
            f"for {EARLY_STOPPING_PATIENCE} consecutive epochs."
        )
        break

print("\nBaseline training finished.")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Final global step: {global_step}")
print(f"Best checkpoint: {best_checkpoint_path}")
print(f"Last checkpoint: {last_checkpoint_path}")
print(f"History: {history_path}")

if torch.cuda.is_available():
    peak_memory_gb = (
        torch.cuda.max_memory_allocated() /
        1024 ** 3
    )

    print(
        f"Peak GPU memory allocated: "
        f"{peak_memory_gb:.2f} GB"
    )

In [ ]:
print("Training history:")

for index, (
    epoch,
    step,
    train_loss,
    val_loss,
    lr,
) in enumerate(
    zip(
        history["epoch"],
        history["global_step"],
        history["train_loss"],
        history["val_loss"],
        history["learning_rate"],
    ),
    start=1,
):
    print(
        f"Epoch {epoch:02d} | "
        f"step={step:04d} | "
        f"train={train_loss:.4f} | "
        f"val={val_loss:.4f} | "
        f"lr={lr:.2e}"
    )

archive_path = shutil.make_archive(
    str(OUTPUT_DIR),
    "zip",
    root_dir=OUTPUT_DIR,
)

print("\nSaved archive:", archive_path)
print("Output files:")

for output_file in sorted(OUTPUT_DIR.iterdir()):
    print("-", output_file.name)